# Introduction to Quantum Computing
<img src="https://raw.githubusercontent.com/FSCJ-FacultyDev/WC26-SPII-QuantumWorkshop/main/images/ibm-quantum.png"
     alt="IBM-QUANTUM"
     width="25%">
<p>(École Polytechnique Fédérale de Lausanne, n.d.)

### Prof. David Singletary
### Florida State College at Jacksonville

# Day 3: Multi-Qubit Systems, Entanglement, and Circuit Simulation
- Current Events
- Review Day 2 Homework
- Multiple Qubits and Circuits (Ch. 4)
- Product States
- Entanglement
- Multi-Qubit States
- Changing Amplitudes with Quantum Transformations
- Example: Target Qubit Pairing (3 Qubits)
- Hands-On: Applying a Gate by Pairing Amplitudes
- Hands-On: Simulating Quantum Circuits


## Current Events

[Google Whitepaper (3/30/26) "Securing Elliptic Curve Cryptocurrencies against Quantum Vulnerabilities:
Resource Estimates and Mitigations"](https://quantumai.google/static/site-assets/downloads/cryptocurrency-whitepaper.pdf)

- Quantum computers could break the security used in today's blockchains
- The math behind systems like Bitcoin's encryption could be solved by quantum algorithms
- Attacks could happen in minutes on advanced quantum hardware
- Transactions could be intercepted and broken while they are being processed
- Smart contracts, staking systems, and even forgotten funds (dormant or abandoned assets) could be at risk
- Fixing this requires both new encryption methods and policy decisions
- Blockchain systems should begin upgrading to quantum-resistant security as soon as possible

### Related Pages
[Safeguarding cryptocurrency by disclosing quantum vulnerabilities responsibly](https://research.google/blog/safeguarding-cryptocurrency-by-disclosing-quantum-vulnerabilities-responsibly)

[Bitcoin bulls scramble for post-quantum protection as Google drops bombshell paper](https://www.coindesk.com/tech/2026/03/31/bitcoin-bulls-scramble-for-post-quantum-protection-as-google-drops-bombshell-paper)

## Day 2 Homework Solution

- Create a “Biased Quantum Coin”
  - Build a single-qubit state where outcomes 0 and 1 are not equally likely, then simulate measurements.

In [ ]:
# Day 2 Homework Solution

# 0. import required modules
from random import choices
from collections import Counter
from math import pi, cos, sin
import numpy as np

# transform state by applying gate
def transform(state, gate):
    result = gate @ np.array(state, dtype=complex)

    # write results back into the original list
    for i in range(len(state)):
        state[i] = result[i]

# define ry gate as a function
def ry(theta):
    return np.array([
        [cos(theta/2), -sin(theta/2)],
        [sin(theta/2),  cos(theta/2)]
    ], dtype=complex)

# print the state
def print_state(state):
    print("Quantum State:")
    print(f"|0⟩ : {state[0]}")
    print(f"|1⟩ : {state[1]}")

# 1. Initialize a qubit to the |0⟩ state
def init_state():
    return [1 + 0j, 0j]

state = init_state()

# 2. Apply one or more gates to create unequal probabilities
# by using a rotation or combination of gates.
# Hint: Try using a rotation gate like ry(θ); different angles change
# how much probability shifts toward |0⟩ or |1⟩ (experiment with θ).
transform(state, ry(pi/3))

# 3. Display the resulting state to observe amplitudes
print_state(state)

# 4. Compute probabilities from amplitudes (squared magnitudes)
probs = [round(float(abs(state[k])**2), 5) for k in range(len(state))]
print("\nProbabilities:")
print(probs)

# 5. Simulate 10 measurements using probabilistic sampling
samples = choices(range(len(state)), probs, k=10)
print("\nSamples (10 shots):")
print(samples)

# 6. Count how many times each outcome appears (10 shots)
# Note: sometimes 1 is shown first; Counter does not sort its output.
# results are returned in the order outcomes are first encountered in the data
print("\nCounts (10 shots):")
for (k, v) in Counter(samples).items():
    print(str(k) + " -> " + str(v))

# 7. Simulate 1000 measurements
samples = choices(range(len(state)), probs, k=1000)

# 8. Count how many times each outcome appears (1000 shots)
print("\nCounts (1000 shots):")
for (k, v) in Counter(samples).items():
    print(str(k) + " -> " + str(v))

# 9. Verify that one outcome occurs more frequently than the other

## Multiple Qubits and Circuits

## Measurement Counts
- When running a quantum computation, the output is a measurement result, a binary string with a digit for each measured qubit.
- Our textbook assumes all qubits are to be measured unless specified otherwise.
- The same computation is repeated and outcome frequencies (counts) are measured.
- The interpretation of the counts is determined by the problem being solved.
- Three examples from Day 1/Chapter 1:
  1. Sampling from probability distributions
  2. Searching for specific outcomes
  3. Estimating the probability of specific outcomes


### Sampling from Probability Distributions
- Each run of a quantum circuit produces one random outcome, so multiple runs are needed to observe the full probability distribution.
- The frequency of outcomes reflects the probabilities determined by the amplitudes of the quantum state.
- Quantum circuits can be designed to encode specific probability distributions, allowing true random sampling useful for simulations, modeling, and random number generation.

### Quantum Computations and Limitations with Python Simulators
- Classical simulators model quantum states and operations, allowing us to develop, test, and inspect quantum programs before running them on real hardware.
- In the Week 1 and 2 workshops we did some simulations of quantum states using simple Python code.
- We have to be cognizant of the limitations of simulation; as discussed previously, it becomes increasingly expensive as the number of qubits grows, because each additional qubit doubles the required computation.

### Two-Qubit State Representation

- A system with two qubits has four possible measurement outcomes: `00`, `01`, `10`, and `11`.
- The state vector contains four amplitudes, one for each outcome.
- The sum of the squared magnitudes of the amplitudes must equal 1.
- A two-qubit state table can be defined with four probabilities (p₀, p₁, p₂, p₃) that add to 1 and four angles (θ₀, θ₁, θ₂, θ₃) defining the direction for each of them.

### How the Amplitudes Are Computed

- The quantum state is created using the following formula for each outcome:

```python
    state = [sqrt(p0) * (cos(theta0) + 1j * sin(theta0)),
             sqrt(p1) * (cos(theta1) + 1j * sin(theta1)),
             sqrt(p2) * (cos(theta2) + 1j * sin(theta2)),
             sqrt(p3) * (cos(theta3) + 1j * sin(theta3))]
```

NOTE: remember that j cannot be used by itself in Python (it is not a predefined constant), it always needs a numeric literal (1 in this case).
- Each value in the list represents the complex amplitude for one measurement outcome.
- p0, p1, p2, p3 are the probabilities associated with each outcome.
- sqrt(p) converts a probability into the magnitude of the amplitude.
- cos(θ) + i sin(θ) determines the direction (phase) of the complex number.
- The resulting list contains the four amplitudes corresponding to the outcomes

  | Index | Outcome |
  |-------|---------|
  | 0     | 00      |
  | 1     | 01      |
  | 2     | 10      |
  | 3     | 11      |

- For the state to be valid, the sum of the squared magnitudes of the amplitudes must equal 1.
- These probabilities represent a system that is definitely in the state 00.
- In the code below, the first outcome has probability 1, and all others have probability 0.
- This is the default starting state for a two-qubit system.
- In Dirac notation, this state is written as the ket |00⟩.

In [ ]:
# here, the state vector is implemented as a list of complex numbers.
from math import sqrt, cos, sin

# probabilities
[p0, p1, p2, p3] = [1, 0, 0, 0]
# angles representing phase of each amplitude
[theta0, theta1, theta2, theta3] = [0, 0, 0, 0]

state = [sqrt(p0) * (cos(theta0) + 1j * sin(theta0)),
         sqrt(p1) * (cos(theta1) + 1j * sin(theta1)),
         sqrt(p2) * (cos(theta2) + 1j * sin(theta2)),
         sqrt(p3) * (cos(theta3) + 1j * sin(theta3))]

# state |00⟩ with amplitude 1, all other outcomes 0
print(state)

## Generating Random 2-Qubit States with Sampling
In the day 2 workshop we used random to do probabilistic sampling using Python's random package
- We can use it here to generate a random, normalized 2-Qubit quantum state by assigning random probabilities and phases to each amplitude.
- A seed is set to ensure the same “random” values are produced each time for consistent results.
- As before, a state table is presented to visualize this.

In [ ]:

import random
from math import pi, sqrt, cos, sin

random.seed(123456789)
probs = [random.random() for _ in range(4)]
total = sum(probs)
probs = [p/total for p in probs]
angles = [random.uniform(0, 2*pi) for _ in range(4)]
state = [sqrt(p)*(cos(a) + 1j*sin(a)) for (p, a) in zip(probs, angles)]
print(state)

- This generates the values represented by the following state table

  <img src="https://raw.githubusercontent.com/FSCJ-FacultyDev/WC26-SPII-QuantumWorkshop/main/images/random-2bit-state.png"
      alt="Random Values for 2 Qubit State"
      width="75%">

## Product States
- A two-qubit state can be formed by combining two independent single-qubit states, known as **product states**, similar to tossing two coins and considering all possible outcomes.
- A combined state is created by taking all pairwise products of the amplitudes from two single-qubit states.
- **When multiplying amplitudes in polar form, magnitudes multiply and phases (angles) add.**
- The resulting two-qubit state can be computed either directly via this multiplication or by using formulas that combine probabilities and phases.
- This lets us build a multi-qubit system by combining simpler pieces we already understand

In [ ]:
from math import pi, sqrt, cos, sin

# cis function from Day 2
def cis(theta):
  return cos(theta) + 1j*sin(theta)

# first state
p = 0.75
theta0 = 0
theta1 = 60/(180/pi)  # convert to radians, required for Python functions
first_state = [sqrt(p)*cis(theta0), sqrt(1-p)*cis(theta1)]
print('first state:')
print([round(amp.real, 5)+1j*round(amp.imag, 5) for amp in first_state])

# second state
q = 0.5
phi0 = 0
phi1 = -120/(180/pi)
second_state = [sqrt(q)*cis(phi0), sqrt(1-q)*cis(phi1)]
print('second state:')
print([round(amp.real, 5)+1j*round(amp.imag, 5) for amp in second_state])

# create a product two-qubit state from the two single-qubit states above
# the new amplitudes will be all the possible products of the two pairs.
new_state = [first_state[0]*second_state[0], first_state[0]*second_state[1],
    first_state[1]*second_state[0], first_state[1]*second_state[1]]
print('product state:')
print([round(amp.real, 5)+1j*round(amp.imag, 5) for amp in new_state])

- which produces the following state table:

  <img src="https://raw.githubusercontent.com/FSCJ-FacultyDev/WC26-SPII-QuantumWorkshop/main/images/figure4.5.png"
        alt="State Table for Product State"
        width="40%">

## Quantum Entanglement
- Not all two-qubit states are product states with independent outcomes.
- Quantum computing uses **entanglement** to enable qubits to work together.
- Changing or measuring one affects the whole system
- Entangled qubits states cannot be described independently of one another,
even when separated by large distances
- Entanglement is essential for meaningful quantum computing, but it introduces tradeoffs, including higher resource demands, system complexity, and reduced coherence.



## Bell States and Entanglement
- Entanglement is a broad concept
- **Bell states** are frequently used as a standard example.
- They are a set of four maximally entangled two-qubit states
- The qubits are fully correlated and cannot be described independently
- Target behavior:
  - Only specific paired outcomes occur
  - Each allowed outcome has equal probability (50%)
- e.g., in the first pair of Bell states the system is constructed so that:
  - Only '00' and '11' can be measured
  - Each occurs with 50% probability
  - '01' and '10' never occur
- This resembles a classical system where two coins always match (both heads or both tails)

### Product States Cannot Produce This Behavior
- In a product (independent) state:
- Joint probabilities are formed by multiplying individual probabilities
- To make '01' and '10' = 0:
  - Each qubit must be fixed (always 0 or always 1)
  - This results in:
  - Only one outcome ('00' or '11'), not both

- Independent qubits cannot produce a 50/50 split of '00' and '11', so the system must be entangled

  <img src="https://raw.githubusercontent.com/FSCJ-FacultyDev/WC26-SPII-QuantumWorkshop/main/images/bell-state-tables.png"
        alt="Bell States State Tables"
        width="75%">

### First Pair of Bell States (|00⟩ and |11⟩)
- bell_state1 = [√0.5, 0, 0,  √0.5]
- bell_state2 = [√0.5, 0, 0, −√0.5]
- Outcomes:
  - Only '00' and '11'
- Probabilities:
  - Each = 0.5
- Difference:
  - A phase (sign) change on the '11' amplitude

### Second Pair of Bell States (|01⟩ and |10⟩)
- bell_state3 = [0, √0.5,  √0.5, 0]
- bell_state4 = [0, √0.5, −√0.5, 0]
- Outcomes:
  - Only '01' and '10'
- Probabilities:
  - Each = 0.5
- Difference:
  - A phase (sign) change on the '10' amplitude

### Bell States Summary
- Restrict outcomes to specific correlated pairs
- Maintain equal probabilities
- Differ only by phase (direction), not probability



## Multi-Qubit States
- An n-qubit system has 2ⁿ possible measurement outcomes.
  - One complex amplitude per outcome
  - Each amplitude is zₖ = aₖ + ibₖ (real + imaginary parts)
- The full quantum state a list of 2ⁿ complex numbers
- Representing a Quantum State in Code
  - A quantum state is stored as a list:
  - Index = outcome (integer form of binary state)
  - Value = complex amplitude
  - e.g.,
```
    state = [(0.09858+0.03637j),
            (0.07478+0.06912j),
            (0.04852+0.10526j),
            (0.00641+0.16322j),
            (-0.12895+0.34953j),
            (0.58403-0.6318j),
            (0.18795-0.08665j),
            (0.12867-0.00506j)]
```
- Valid Quantum State Requirements
  - Length must be a power of 2
  - Sum of squared magnitudes must equal 1
```
      from math import log2, ceil, floor

      def is_power_of_two(m):
          return ceil(log2(m)) == floor(log2(m))

      def prepare_state(*a):
          state = [a[k] for k in range(len(a))]
          assert(is_power_of_two(len(state)))
          assert(sum(abs(state[k])**2 for k in range(len(state))) == 1)
          return state
```
- Viewing the State (State Table)
  - Display amplitudes with their corresponding outcomes:
```
print([[k, state[k]] for k in range(len(state))])
```
- Output structure:
  - [outcome, amplitude]
### From Amplitudes to Probabilities and Directions
- Probability: pₖ = |zₖ|²
- Direction (phase angle) is derived using atan2(imag, real)
```
    from math import atan2, pi

    table = [
        [
            k,
            round(atan2(state[k].imag, state[k].real) / (2*pi) * 360, 5),
            round(abs(state[k])**2, 5)
        ]
        for k in range(len(state))
    ]
```
- Each row includes [outcome, direction (degrees), probability]
- An expanded State Table can be coded by ombining everything into one view
```
expanded_table = [
    [
        k,
        state[k],
        round(atan2(state[k].imag, state[k].real) / (2*pi) * 360, 5),
        round(abs(state[k]), 5),
        round(abs(state[k])**2, 5)
    ]
    for k in range(len(state))
]
```
- Each row includes:
  - Outcome
  - Amplitude
  - Direction (phase)
  - Magnitude
  - Probability

### From Probabilities and Directions to Amplitudes
- Reverse the process:
  - Magnitude = √probability
  - Use cosine/sine to reconstruct complex number
```
from math import sqrt, cos, sin, pi

table2 = [
    [
        row[0],
        (
            sqrt(row[2]) * cos(row[1] / (180/pi)) +
            sqrt(row[2]) * sin(row[1] / (180/pi)) * 1j
        )
    ]
    for row in table
]
```
- This confirms that probabilities and directions fully define the state
- It also reconfirms what we saw before about the advantages of simulation:
  - you can directly set amplitudes in the code (not possible on real quantum hardware)


In [ ]:
from math import log2, ceil, floor, atan2, pi, sqrt, cos, sin

def is_close(a, b, tol=1e-9):
    return abs(a - b) <= tol


def is_power_of_two(m):
    return m > 0 and ceil(log2(m)) == floor(log2(m))

def prepare_state(*amplitudes):
    """
    Validate and return a quantum state represented as a list
    of complex amplitudes.
    """
    state = list(amplitudes)

    assert is_power_of_two(len(state)), \
        "State length must be a power of 2."

    total_probability = sum(abs(state[k]) ** 2 for k in range(len(state)))
    assert is_close(total_probability, 1.0, tol=1e-4), \
        "Sum of squared magnitudes must equal 1."

    return state

def amplitude_table(state):
    """
    Return [outcome, amplitude] rows.
    """
    return [[k, state[k]] for k in range(len(state))]

def direction_probability_table(state):
    """
    Return [outcome, direction_in_degrees, probability] rows.
    """
    return [
        [
            k,
            round(atan2(state[k].imag, state[k].real) / (2 * pi) * 360, 5),
            round(abs(state[k]) ** 2, 5)
        ]
        for k in range(len(state))
    ]

def expanded_table(state):
    """
    Return [outcome, amplitude, direction_in_degrees, magnitude, probability] rows.
    """
    return [
        [
            k,
            state[k],
            round(atan2(state[k].imag, state[k].real) / (2 * pi) * 360, 5),
            round(abs(state[k]), 5),
            round(abs(state[k]) ** 2, 5)
        ]
        for k in range(len(state))
    ]

def reconstruct_amplitudes(table):
    """
    Rebuild amplitudes from [outcome, direction_in_degrees, probability].
    """
    return [
        [
            row[0],
            (
                round(sqrt(row[2]) * cos(row[1] / (180 / pi)), 5) +
                round(sqrt(row[2]) * sin(row[1] / (180 / pi)), 5) * 1j
            )
        ]
        for row in table
    ]

def print_table(title, rows):
    print(f"\n{title}")
    print("-" * len(title))
    for row in rows:
        print(row)

def main():
    # Example 3-qubit state from the text
    amplitude_list = [
        (0.09858 + 0.03637j),
        (0.07478 + 0.06912j),
        (0.04852 + 0.10526j),
        (0.00641 + 0.16322j),
        (-0.12895 + 0.34953j),
        (0.58403 - 0.6318j),
        (0.18795 - 0.08665j),
        (0.12867 - 0.00506j)
    ]

    print("Multi-Qubit State Demo")
    print("======================")
    print("This program demonstrates how a quantum state can be:")
    print("1. Stored as a list of complex amplitudes")
    print("2. Validated as a proper state")
    print("3. Converted to direction/probability form")
    print("4. Reconstructed from that form")

    # Validate and prepare the state
    state = prepare_state(*amplitude_list)

    print(f"\nNumber of amplitudes: {len(state)}")
    print(f"Number of qubits: {int(log2(len(state)))}")
    print(f"Total probability: {sum(abs(a) ** 2 for a in state):.5f}")

    # Table 1: outcome + amplitude
    amp_table = amplitude_table(state)
    print_table("Amplitude Table", amp_table)

    # Table 2: outcome + direction + probability
    table1 = direction_probability_table(state)
    print_table("Direction/Probability Table", table1)

    # Table 3: expanded state table
    full_table = expanded_table(state)
    print_table("Expanded State Table", full_table)

    # Table 4: reconstruct amplitudes
    reconstructed = reconstruct_amplitudes(table1)
    print_table("Reconstructed Amplitudes", reconstructed)

    print("\nNotes")
    print("-----")
    print("- Each list index corresponds to one measurement outcome.")
    print("- Each value is that outcome's complex amplitude.")
    print("- The probability of an outcome is |amplitude|^2.")
    print("- The direction is the phase angle in degrees.")
    print("- Small differences in reconstructed amplitudes are due to rounding.")

main()

  <img src="https://raw.githubusercontent.com/FSCJ-FacultyDev/WC26-SPII-QuantumWorkshop/main/images/figure4.14.png"
        alt="Table for Quantum State with n Qubits"
        width="75%">

  **Quantum State with n = 3 Qubits**

### Simulating Multi-Qubit States in Python
- Multi-qubit quantum states can be represented as Python lists
  - Length = 2ⁿ (for n qubits)
  - Each index corresponds to a measurement outcome
  - Each value is the amplitude for that outcome
- Initializing a Multi-Qubit State
- Quantum systems typically start in a default state, with all qubits initialized to |0⟩, i.e., a 100% probability of measuring 0…0
- Initialization sets a known starting point for computation
- All quantum operations (gates) transform this state afterward
- To simulate, use an init_state(n) function
```
  def init_state(n):
      state = [0 for _ in range(2 ** n)]  # create list of size 2^n
      state[0] = 1                        # set amplitude for |00...0⟩
      return state
```
- The initialized state has amplitude 1 at index 0 (|00...0⟩), and amplitude 0 everywhere else
- This represents a system with complete certainty in one outcome
- e.g., for a two-qubit system
```
  state = init_state(2)
  print(state)
```
- Outputs [1, 0, 0, 0]
- |00⟩ has probability 1, |01⟩, |10⟩, |11⟩ have probability 0



## Changing Amplitudes with Quantum Transformations
- We have seen that quantum computation uses gate transformations
- Gates modify amplitudes
  - Updated amplitudes lead to updated probabilities
- Single-Qubit Gate Behavior
  - A single-qubit state has 2 amplitudes
  - A gate recombines these using two equations
  - This produces a new pair of amplitudes
- Multi-Qubit Gate Behavior
  - A multi-qubit state has 2ⁿ amplitudes
  - The same gate equations are applied, but across multiple pairs of amplitudes
  - Each pair is recombined independently
- Quantum gate transformations are applied to a selected qubit in
a system called the **target qubit**
- The target qubit determines which amplitudes get paired together, and how the state is recombined

### Selecting Pairs of Amplitudes Based on the Target Qubit
- The target qubit (t) determines how amplitudes are grouped into pairs for a gate operation
- Pairing determines which amplitudes get recombined together when a gate is applied to a specific qubit.
- To form pairs:
  - Convert outcomes to binary strings
  - Pair amplitudes whose outcomes differ only at the target qubit position
  - Qubits are indexed right to left:
    - Rightmost bit = qubit 0
    - Next = qubit 1, etc.
- e.g., a 3-qubit system has 8 outcomes
  - '000', '001', '010', '011', '100', '101', '110', '111'
  - If t = 0 (rightmost qubit):
  - Pair outcomes that differ only in the last bit
  - '000' ↔ '001'
  - '010' ↔ '011'
  - '100' ↔ '101'
  - '110' ↔ '111'

  <img src="https://raw.githubusercontent.com/FSCJ-FacultyDev/WC26-SPII-QuantumWorkshop/main/images/figure4.15.png"
        alt="3-Qubit System Outcome Pairs"
        width="75%">

  **Outcome pairs for Single-Qubit Gate to 3-Qubit System**

- Each pair is recombined using the same single-qubit gate equations
- When simulated, pairs are processed sequentially
- On real quantum hardware, the transformation is applied simultaneously, not step-by-step


### Example: Target Qubit Pairing (3 Qubits)
- Create a valid 3-qubit quantum state
- Display amplitudes
- Group amplitudes into pairs based on a target qubit
- Show how pairing changes depending on the target t
  - The target qubit determines pairing
  - Pairs differ in exactly one bit position
  - We use bitwise XOR to flip that bit:
    - j = i ^ (1 << target)
**t = 0 (rightmost bit)**
```
000 <-> 001
010 <-> 011
100 <-> 101
110 <-> 111
```
**t = 1 (middle bit)**
```
000 <-> 010
001 <-> 011
100 <-> 110
101 <-> 111
```
**t = 2 (leftmost bit)**
```
000 <-> 100
001 <-> 101
010 <-> 110
011 <-> 111
```

In [ ]:
# create valid multi-qubit quantum state and group amplitudes into pairs.
# outcomes differ only at a chosen target qubit using bitwise operations.

from math import sqrt

# -------------------------------
# Utility Functions
# -------------------------------
def is_power_of_two(m):
    return m > 0 and (m & (m - 1)) == 0

def prepare_state(*amplitudes):
    state = list(amplitudes)

    assert is_power_of_two(len(state)), \
        "State length must be a power of 2."

    total_probability = sum(abs(a) ** 2 for a in state)
    assert abs(total_probability - 1.0) < 1e-6, \
        "State must be normalized."

    return state

def format_binary(index, n):
    return format(index, f'0{n}b')

# -------------------------------
# Pairing Logic
# -------------------------------
def get_pairs(state, target):
    """
    Return list of index pairs where outcomes differ
    only at the target qubit.
    """
    n = len(state)
    num_qubits = int(n).bit_length() - 1

    visited = set()
    pairs = []

    for i in range(n):
        if i in visited:
            continue

        # flip the target qubit using XOR
        j = i ^ (1 << target)

        if j not in visited:
            pairs.append((i, j))
            visited.add(i)
            visited.add(j)

    return pairs

def display_pairs(state, target):
    n = len(state)
    num_qubits = int(n).bit_length() - 1

    print(f"\nTarget qubit t = {target}")
    print("-" * 40)

    pairs = get_pairs(state, target)

    for i, j in pairs:
        b1 = format_binary(i, num_qubits)
        b2 = format_binary(j, num_qubits)

        print(f"{b1} <-> {b2}")
        print(f"  amplitudes: {state[i]} , {state[j]}")
        print()

# -------------------------------
# Example State (3 qubits -> 8 amplitudes)
# -------------------------------
amp = 1 / sqrt(8)

state = prepare_state(
    amp, amp, amp, amp,
    amp, amp, amp, amp
)

print("Quantum State (index -> amplitude):")
for i, amp in enumerate(state):
    print(f"{format_binary(i, 3)} : {amp}")

# -------------------------------
# Show Pairing for Each Target Qubit
# -------------------------------
for t in range(3):
    display_pairs(state, t)

## Hands-On Exercise: Applying a Gate by Pairing Amplitudes

- We already know how to apply a gate using matrix multiplication:
```
        {new pair} = G @ [a, b]
```

- The key question for multi-qubit systems is **which amplitudes form the pair [a, b]?**

- Remember that a gate acts on a target qubit
- The target qubit determines how amplitudes are paired
- Each pair consists of outcomes that differ in exactly one bit (the target bit)
- We find the matching index using:
```
        j = i ^ (1 << target)
```

In [ ]:
import numpy as np
from math import sqrt

# Gates
H = (1 / sqrt(2)) * np.array([
    [1,  1],
    [1, -1]
], dtype=complex)

X = np.array([
    [0, 1],
    [1, 0]
], dtype=complex)

def get_pairs(n, target):
    """
    Return index pairs whose values differ only at the target bit, found by
    flipping the target bit using index XOR.
    """
    visited = set()
    pairs = []

    for i in range(n):
        if i in visited:
            continue

        j = i ^ (1 << target)

        if j not in visited:
            pairs.append((i, j))
            visited.add(i)
            visited.add(j)

    return pairs

def apply_gate(state, target, gate):
    """
    Apply gate to state by pairing amplitudes whose indices differ only at
    the target bit (found via index flipping) and applying gate @ [a, b].
    """
    new_state = state.copy()
    pairs = get_pairs(len(state), target)

    for i, j in pairs:
        new_state[i], new_state[j] = gate @ np.array([state[i], state[j]])

    return new_state

def print_state(state):
    n = len(state).bit_length() - 1
    for i, amp in enumerate(state):
        print(f"{format(i, f'0{n}b')} : {amp}")

# uniform 3-qubit state
amp = 1 / sqrt(8)
state = np.array([amp] * 8, dtype=complex)

print("Original state:")
print_state(state)

target = 0
new_state = apply_gate(state, target, H)

print("\nAfter applying H to target qubit 0:")
print_state(new_state)

- Which amplitudes were paired for target = 0?

## Try It
### Change the Target
- Try changing target to 1 and then 2
- How do the pairings change?
### Apply Different Gates
- Replace H with X
  - What happens to each pair?
  - Do values change or just move?
- Try applying `H` twice, e.g.
```
        state2 = apply_gate(state, target, H)
        state3 = apply_gate(state2, target, H)
```


### Encoding a Uniform Distribution in a Multi-Qubit System
- All outcomes have equal probability => all amplitudes must have equal magnitude
- Initial state (3 qubits)
- Start in default state:
```
    state = init_state(3)
    # [1, 0, 0, 0, 0, 0, 0, 0]
```
- Only '000' has nonzero amplitude
- Applying Hadamard Gates (Building the Distribution)
- Each gate recombines amplitudes in pairs based on target qubit
#### Apply H to target = 0
```
    transform(state, 0, h)
```
- Affects pair: '000' ↔ '001'
- Result: [0.7071, 0.7071, 0, 0, 0, 0, 0, 0]
#### Apply H to target = 1
```
    state = init_state(3)
    transform(state, 1, h)
```
- Affects pair: '000' ↔ '010'
- Result: [0.7071, 0, 0.7071, 0, 0, 0, 0, 0]
### Full Uniform Distribution (All Qubits)
#### Apply Hadamard to every qubit:
```
state = init_state(3)
transform(state, 0, h)
transform(state, 1, h)
transform(state, 2, h)
```
- Final state:
```
    [0.3536, 0.3536, 0.3536, 0.3536,
     0.3536, 0.3536, 0.3536, 0.3536]
```
- Applying H to each qubit spreads amplitude across all outcomes
- Final amplitudes have equal magnitude: 1/√(2ⁿ)
- This produces a uniform probability distribution
- Pairing and repeated recombination is how the distribution is built step-by-step in simulation



In [ ]:
import numpy as np
from math import sqrt, log2

# simulate encoding a uniform distribution in a multi-qubit system

# start from |000⟩ (single nonzero amplitude)
# apply Hadamard gates to target qubits
# pair amplitudes by target bit and recombine via matrix multiplication
# repeated application spreads amplitude evenly across all outcomes

# H gate
H = (1 / sqrt(2)) * np.array([
    [1,  1],
    [1, -1]
], dtype=complex)

# -------------------------------
# State Utilities
# -------------------------------

# return the default n-qubit state |00...0>
def init_state(num_qubits):
    state = np.zeros(2 ** num_qubits, dtype=complex)
    state[0] = 1
    return state

# return the binary outcome string for an index, padded to num_qubits bits
def format_binary(index, num_qubits):
    return format(index, f"0{num_qubits}b")

# print a quantum state as outcome -> amplitude
def print_state(state, title=None, decimals=4):
    if title:
        print(title)

    num_qubits = int(log2(len(state)))
    for i, amp in enumerate(state):
        print(f"{format_binary(i, num_qubits)} : {np.round(amp, decimals)}")
    print()

# print the state as a rounded list of amplitudes
def print_state_vector(state, title=None, decimals=4):
    if title:
        print(title)
    rounded = [complex(round(a.real, decimals), round(a.imag, decimals)) for a in state]
    print(rounded)
    print()

# return a list of measurement probabilities for the state
def probability_distribution(state, decimals=4):
    return [float(round(abs(a) ** 2, decimals)) for a in state]

# -------------------------------
# Pairing / Transformation Logic
# -------------------------------

# return index pairs whose values differ only at the target bit
def get_pairs(n, target):
    visited = set()
    pairs = []

    for i in range(n):
        if i in visited:
            continue

        j = i ^ (1 << target)   # flip the target bit

        visited.add(i)
        visited.add(j)

        pairs.append((min(i, j), max(i, j)))

    return sorted(pairs)

# apply a single-qubit gate to the chosen target qubit of a multi-qubit state
def transform(state, target, gate):
    n = len(state)
    pairs = get_pairs(n, target)
    new_state = state.copy()

    for i, j in pairs:
        pair = np.array([state[i], state[j]], dtype=complex)
        new_pair = gate @ pair
        new_state[i], new_state[j] = new_pair

    return new_state

# display the amplitude pairs affected by a gate on the selected target qubit
def show_pairings(state, target):
    n = len(state)
    num_qubits = int(log2(n))
    pairs = get_pairs(n, target)

    print(f"Target qubit t = {target}")
    print("-" * 40)

    for i, j in pairs:
        left = format_binary(i, num_qubits)
        right = format_binary(j, num_qubits)
        print(f"{left} <-> {right}")
        print(f"  amplitudes: {state[i]} , {state[j]}")
        print()


# -------------------------------
# Demonstration
# -------------------------------

# initial state (3 qubits)
state = init_state(3)
print_state_vector(state, "Initial state (3 qubits):")
print_state(state, "Expanded view of initial state:")

# show the initial pairing for target = 0
show_pairings(state, 0)

# apply H to target = 0
state_t0 = transform(state, 0, H)
print_state_vector(state_t0, "After applying H to target = 0:")
print_state(state_t0, "Expanded view:")

# reset and apply H to target = 1
state = init_state(3)
show_pairings(state, 1)

state_t1 = transform(state, 1, H)
print_state_vector(state_t1, "After applying H to target = 1:")
print_state(state_t1, "Expanded view:")

# build the full uniform distribution by applying H to every qubit
state = init_state(3)
print_state_vector(state, "Reset to initial state:")

for target in [0, 1, 2]:
    print(f"Applying H to target = {target}")
    show_pairings(state, target)
    state = transform(state, target, H)
    print_state_vector(state, f"State after H on target = {target}:")

# final results
print_state(state, "Final state after applying H to all 3 qubits:")
print("Final probability distribution:")
print(probability_distribution(state))
print()

expected_amplitude = 1 / sqrt(2 ** 3)
print(f"Expected equal amplitude magnitude: 1/sqrt(2^3) = {round(expected_amplitude, 4)}")
print(f"Actual amplitude magnitudes: {[float(round(abs(a), 4)) for a in state]}")
print()

### Controlled Quantum Transformations
- Quantum gates normally change all amplitudes at once ( quantum parallelism)
- This enables speedups but also limits fine control
- **Controlled gates** restrict transformations to specific amplitudes
- A controlled gate acts only when the control qubit is 1  
- Pairing still depends on the target qubit
- Only pairs that differ in the target position AND have 1 in the control position are affected
  - All other amplitudes remain unchanged

  <img src="https://raw.githubusercontent.com/FSCJ-FacultyDev/WC26-SPII-QuantumWorkshop/main/images/figure4.25-26.png"
        alt="Controlled Pairings: Target 0, Control 2 vs Target 2, Control 1"
        width="75%">

    **Controlled Pairings:  
    Target 0, Control 2 vs Target 2, Control 1**

- Controlled gates reduce the number of affected pairs  
  - Each control qubit cuts the number of active pairs in half (in simulation)
- The gate math (recombination of amplitudes) stays the same  
- Only the subset of pairs being updated changes
  - Target = where the change happens  
  - Control = when the change is allowed
- On real hardware:
  - Controlled gates are more expensive
  - They can increase errors (reduce coherence)
- Some ways these are addressed:
  - minimizing use of controlled gates in circuit design   
  - decomposing controlled gates into simpler gates  
  - using hardware-native gate sets

### Multi-Control Gates

- A **multi-control gate** applies a transformation only when *all* control qubits are 1  
- It extends the idea of a single controlled gate to multiple conditions  
- The **target qubit** is only affected when every control condition is satisfied  
- In simulation:
  - Each added control reduces the number of active amplitude pairs by half  
  - Most of the state remains unchanged  

#### Example: Multi-Control X (MCX)

- Two control qubits: q₀ and q₁  
- One target qubit: q₂  
- This acts like a logical **AND**:
  - Flip q₂ **only if** q₀ = 1 **and** q₁ = 1  

| Input State | Output State |
|-------------|------------|
| 000         | 000        |
| 010         | 010        |
| 100         | 100        |
| 110         | 111  ← flip occurs |

- In circuits, this is often written as **mcx([q0, q1], q2)**
- Conceptually:
  - **Controls = condition**
  - **Target = where the change happens**

## Hands-On: Simulating Quantum Circuits
- Let's move from a functional style to an object-oriented approach
- This simulation structure closely matches real quantum frameworks; it mirrors Qiskit so there is an easy transition to quantum hardware
- A Circuit object stores:
  - quantum state (amplitudes)
  - list of transformations (gates)
- The results include a Python dictionary (key, value) of outcome, count
- As before, counts approximate true probabilities over many run




### Start by setting up the textbook repo for imports

In [ ]:
# ---- setup for clone/repo imports ----

import os
import sys
import subprocess
import importlib

REPO_URL = "https://github.com/learnqc/code.git"
REPO_DIR = "/content/code"
SRC_DIR = f"{REPO_DIR}/src"

# Install required pip package(s)
subprocess.run(
    ["pip", "install", "-q", "sty"],
    check=True
)

# Clone repo if needed
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

# Make src importable
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Clear any stale imports
importlib.invalidate_caches()

print("Setup complete")

### Quantum Registers
- Qubits are grouped into registers
- Each qubit has an index
- This supports indexing (q[0]) and iteration and helps manage multi-qubit systems cleanly

In [ ]:
# QuantumRegister class represents a group of qubits and
# maps local indices to global circuit indices
class QuantumRegister:

    # initialize a register with a given number of qubits and starting offset
    # size = number of qubits in this register
    # shift = starting index in the full system
    def __init__(self, size, shift=0):
        self.size = size
        self.shift = shift

    # return the global index/list of indices for a given local position
    # converts local index to global index using shift
    # (if key is a slice, returns a list of mapped indices)
    def __getitem__(self, key):
        if isinstance(key, slice):
            return [self[ii] for ii in range(*key.indices(len(self)))]
        elif isinstance(key, int):
            if key < 0:
                key += len(self)
            assert(0 <= key < self.size)
            return self.shift + key

    # return the number of qubits in the register
    def __len__(self):
        return self.size

    # iterate over all qubit indices in this register
    # returns all qubit indices with shift applied
    def __iter__(self):
        return list([self.shift + i for i in range(self.size)])

    # Return qubit indices in reverse order
    def __reversed__(self):
        return list([self.shift + i for i in range(self.size)[::-1]])



### Quantum Transformations
- Each operation includes:
  - gate (matrix)
  - target qubit
  - optional control qubits
- Stored as objects in a circuit
- Executed later in sequence


In [ ]:
# QuantumTransformation class represents a single quantum operation
# (gate + target + optional controls)
class QuantumTransformation:
    # initialize transformation with gate, target qubit, controls, and metadata
    def __init__(self, gate, target, controls=[], name=None, arg=None):
        self.gate = gate
        self.target= target
        self.controls = controls
        self.name = name
        self.arg = arg

    # return a readable string describing the transformation
    def __str__(self):
      if self.arg is not None:
          arg_str = round(self.arg, 2)
      else:
          arg_str = ""

      return (
          rf'{self.name} {arg_str} '
          f'{self.controls} {self.target}'
    )

    # create a copy of this transformation object
    def __copy__(self):
        return QuantumTransformation(
            self.gate,
            self.target,
            self.controls,
            self.name,
            self.arg
        )

### Quantum Circuits
- A circuit is a sequence of gate applications
- Key steps:
  - initialize state
  - add transformations (H, X, RY, CX, etc.)
  - run circuit → apply all gates
- `run()`:
  - applies transformations in order
  - updates state
  - clears transformation list


In [ ]:
# QuantumCircuit class represents a quantum circuit that stores state
# and a sequence of gate operations
class QuantumCircuit:

    # initialize circuit registers, assign global indices, set initial state
    def __init__(self, *args):
        bits = 0
        regs = []
        for register in args:
            register.shift = bits
            bits += register.size
            regs.append(register.size)

        self.state = init_state(bits)
        self.transformations = []
        self.regs = regs
        self.reports = {}

    # replace current state with a new state vector
    def initialize(self, state):
        self.state = state

    # add X (NOT) gate on target qubit t
    def x(self, t):
        self.transformations.append(
           QuantumTransformation(x, t, [], 'x'))

    # add Hadamard gate on target qubit t
    def h(self, t):
        self.transformations.append(
           QuantumTransformation(h, t, [], 'h'))

    # add Ry rotation gate with angle theta on target qubit t
    def ry(self, theta, t):
        self.transformations.append(
           QuantumTransformation(ry(theta), t, [], 'ry', theta))

    # add controlled-X gate with control c and target t
    def cx(self, c, t):
        self.transformations.append(
           QuantumTransformation(x, t, [c], 'x'))

    # add multi-controlled X gate with controls cs and target t
    def mcx(self, cs, t):
        self.transformations.append(
           QuantumTransformation(x, t, cs, 'x'))

    # run circuit and return final state and sampled measurement counts
    def measure(self, shots=0):
        state = self.run()
        samples = measure(state, shots)
        return {'state vector': state, 'counts': samples}

    # apply all stored transformations in order and update the state
    def run(self):
        for tr in self.transformations:
            cs = tr.controls
            if len(cs) == 0:
                transform(self.state, tr.target, tr.gate)
            elif len(cs) == 1:
                c_transform(self.state, cs[0], tr.target, tr.gate)
            else:
                mc_transform(self.state, cs, tr.target, tr.gate)
        self.transformations = []
        return self.state

### Example Circuit (3 Qubits)
- Apply:
  - H on qubit 0
  - H on qubit 1
  - multi-controlled X on qubit 2
- Result:
  - specific amplitudes become nonzero
- Measurement:
  - repeated runs produce counts matching probabilities

- **Note:** before we look at the example's main code, in the next code block the core simulation code is included explicity instead of importing it so we can walk through it step by step and understand how the simulator actually works.
  - Also note the imports from ch03 modules (gates and utility functions) which are concepts we have already seen, so we won’t repeat that code here

In [ ]:
# from textbook repo: ch4/sim_core.py
from math import log2, ceil, floor
from random import choices
from collections import Counter
from ch03.util import *
from ch03.sim_core import *
from ch03.sim_gates import *

# return True if m is a power of two
def is_power_of_two(m):
    return ceil(log2(m)) == floor(log2(m))

# create a valid quantum state from amplitudes and verify normalization
def prepare_state(*a):
    state = [a[k] for k in range(len(a))]
    assert(is_power_of_two(len(state)))
    assert(is_close(sum([abs(state[k])**2 for k in range(len(state))]), 1.0))
    return state

# initialize an n-qubit state with all amplitude in |0...0⟩
def init_state(n):
    state = [0 for _ in range(2 ** n)]
    state[0] = 1
    return state

# check if bit k is set (1) in integer m
def is_bit_set(m, k):
    return m & (1 << k) != 0

# generate index pairs by checking target bit directly
def pair_generator_check_digit(n, t):
    distance = int(2 ** t)

    for k0 in range(2 ** n):
        if not is_bit_set(k0, t):
            k1 = k0 + distance
            yield k0, k1

# generate index pairs by combining prefix and suffix blocks
def pair_generator_concatenate(n, t):
    distance = int(2 ** t)
    suffix_count = int(2 ** t)
    prefix_count = int(2 ** (n - t - 1))

    for p in range(prefix_count):
        for s in range(suffix_count):
            k0 = p * suffix_count*2 + s
            k1 = k0 + distance
            yield k0, k1

# generate index pairs using repeating chunk patterns
def pair_generator_pattern(n, t):
    distance = int(2 ** t)

    for j in range(2 ** (n-t-1)):
        for k0 in range(2*j*distance, (2*j+1)*distance):
            k1 = k0 + distance
            yield k0, k1

# set the active pair generation strategy
pair_generator = pair_generator_concatenate

# apply a gate to a single pair of amplitudes
def process_pair(state, gate, k0, k1):
    x = state[k0]
    y = state[k1]
    state[k0] = x * gate[0][0] + y * gate[0][1]
    state[k1] = x * gate[1][0] + y * gate[1][1]

# apply a gate to all pairs for target qubit t
def transform(state, t, gate):
    n = int(log2(len(state)))
    for (k0, k1) in pair_generator(n, t):
        process_pair(state, gate, k0, k1)

# apply a controlled gate when control qubit c is 1
def c_transform(state, c, t, gate):
    n = int(log2(len(state)))
    for (k0, k1) in filter(lambda p: is_bit_set(p[0], c), pair_generator(n, t)):
        process_pair(state, gate, k0, k1)

# apply a multi-controlled gate when all control qubits are 1
def mc_transform(state, cs, t, gate):
    assert t not in cs
    n = int(log2(len(state)))

    # nested function: return True if all control qubits are 1 for this pair
    def controls_active(p):
        return all([is_bit_set(p[0], c) for c in cs])

    for (k0, k1) in filter(controls_active, pair_generator(n, t)):
        process_pair(state, gate, k0, k1)

# simulate measurement by sampling outcomes based on probabilities
def measure(state, shots):
    probs = [abs(state[k])**2 for k in range(len(state))]
    outcomes = range(len(state))
    samples = choices(outcomes, probs, k=shots)

    counts = {}
    for (k, v) in Counter(samples).items():
        counts[k] = v
    return counts

- The following block is the main program that uses everything we just built

In [ ]:
q = QuantumRegister(3)          # create a register with 3 qubits
qc = QuantumCircuit(q)          # create a circuit using the register

qc.h(q[0])                      # apply H to qubit 0 (create superposition)
qc.h(q[1])                      # apply H to qubit 1 (create superposition)
qc.mcx([q[0], q[1]], q[2])      # apply multi-controlled X (AND) on qubit 2)

state = qc.run()                # execute all gates and update the state
print_state_table(state)        # display the resulting state amplitudes
samples = measure(state, 1000)  # sample measurement outcomes over 1000 shots
print(samples)                  # print counts of observed outcomes

### Try It
- Change the control/target configuration
  - Try different controls and targets, e.g. qc.mcx([q[1], q[2]], q[0])
  - Observe how the output states that flip change
  - Which input combinations now trigger the X?

<details>
<summary><b>Click here for answer</b></summary>

```
q = QuantumRegister(3)          # create a register with 3 qubits
qc = QuantumCircuit(q)          # create a circuit using the register

qc.h(q[1])                      # apply H to qubit 1 (create superposition)
qc.h(q[2])                      # apply H to qubit 2 (create superposition)
qc.mcx([q[1], q[2]], q[0])      # flip q0 only when q1 = 1 and q2 = 1

state = qc.run()                # execute all gates and update the state
print_state_table(state)        # display the resulting state amplitudes
samples = measure(state, 1000)  # sample measurement outcomes over 1000 shots
print(samples)                  # print counts of observed outcomes
```
- With **qc.mcx([q[1], q[2]], q[0])**, the gate applies X to q₀ only when q₁ = 1 and q₂ = 1.  
- In this circuit, the only triggering input state is `110`, and the transformation flips:
  - 110 -> 111
</details>

In [ ]:
q = QuantumRegister(3)          # create a register with 3 qubits
qc = QuantumCircuit(q)          # create a circuit using the register

qc.h(q[1])                      # apply H to qubit 1 (create superposition)
qc.h(q[2])                      # apply H to qubit 2 (create superposition)
qc.mcx([q[1], q[2]], q[0])      # flip q0 only when q1 = 1 and q2 = 1
state = qc.run()                # execute all gates and update the state
print_state_table(state)        # display the resulting state amplitudes
samples = measure(state, 1000)  # sample measurement outcomes over 1000 shots
print(samples)                  # print counts of observed outcomes

### Try It
- Modify the initial superposition
  - Remove one Hadamard (e.g. only qc.h(q[0])) or add qc.h(q[2])
  - Compare the state table and measurement results
  - How does changing the input distribution affect when the MCX activates?

<details>
<summary><b>Click here for answer</b></summary>

- The MCX gate only activates for basis states where all control qubits are 1
- In the original circuit, the controls are q₀ and q₁, so the gate can act only on states that include q₀ = 1 and q₁ = 1
- The Hadamard gates determine which basis states have nonzero amplitude before the MCX is applied
- If you use only:
  - **qc.h(q[0])**
- then the state before MCX contains only:
  - 000
  - 001
- Since q₁ is always 0 in those states, the MCX condition is never satisfied
- Result:
  - the MCX does nothing
  - the state table stays the same after MCX
  - measurement results show outcomes only from that smaller superposition

```
# Variation 1: only qc.h(q[0])

q = QuantumRegister(3)          # create a register with 3 qubits
qc = QuantumCircuit(q)          # create a circuit using the register

qc.h(q[0])                      # apply H only to qubit 0
qc.mcx([q[0], q[1]], q[2])      # flip q2 only when q0 = 1 and q1 = 1

state = qc.run()                # execute all gates and update the state
print("Variation 1: only qc.h(q[0])")
print_state_table(state)        # display the resulting state amplitudes
samples = measure(state, 1000)  # sample measurement outcomes over 1000 shots
print(samples)                  # print counts of observed outcomes
```

- If you add:
  - **qc.h(q[2])**
- then more basis states appear in the input superposition
- But the MCX still activates only on states where q₀ = 1 and q₁ = 1
- So adding superposition on another qubit does not change the control rule
- It only changes whether the triggering states are present, and with what amplitudes

```
# Variation 2: add qc.h(q[2])

q = QuantumRegister(3)          # create a register with 3 qubits
qc = QuantumCircuit(q)          # create a circuit using the register

qc.h(q[0])                      # apply H to qubit 0
qc.h(q[1])                      # apply H to qubit 1
qc.h(q[2])                      # also apply H to qubit 2
qc.mcx([q[0], q[1]], q[2])      # flip q2 only when q0 = 1 and q1 = 1

state = qc.run()                # execute all gates and update the state
print("Variation 2: add qc.h(q[2])")
print_state_table(state)        # display the resulting state amplitudes
samples = measure(state, 1000)  # sample measurement outcomes over 1000 shots
print(samples)                  # print counts of observed outcomes
```

- In a nutshell
  - changing the initial superposition changes which input states exist before the MCX
  - the MCX activates only if the superposition includes states that satisfy the control condition
  - if those states are missing, the MCX has no visible effect
  - if those states are present, the MCX changes only those parts of the state
</details>

In [ ]:
# Variation 1: only qc.h(q[0])

q = QuantumRegister(3)          # create a register with 3 qubits
qc = QuantumCircuit(q)          # create a circuit using the register

qc.h(q[0])                      # apply H only to qubit 0
qc.mcx([q[0], q[1]], q[2])      # flip q2 only when q0 = 1 and q1 = 1

state = qc.run()                # execute all gates and update the state
print("Variation 1: only qc.h(q[0])")
print_state_table(state)        # display the resulting state amplitudes
samples = measure(state, 1000)  # sample measurement outcomes over 1000 shots
print(samples)                  # print counts of observed outcomes

In [ ]:
# Variation 2: add qc.h(q[2])

q = QuantumRegister(3)          # create a register with 3 qubits
qc = QuantumCircuit(q)          # create a circuit using the register

qc.h(q[0])                      # apply H to qubit 0
qc.h(q[1])                      # apply H to qubit 1
qc.h(q[2])                      # also apply H to qubit 2
qc.mcx([q[0], q[1]], q[2])      # flip q2 only when q0 = 1 and q1 = 1

state = qc.run()                # execute all gates and update the state
print("Variation 2: add qc.h(q[2])")
print_state_table(state)        # display the resulting state amplitudes
samples = measure(state, 1000)  # sample measurement outcomes over 1000 shots
print(samples)                  # print counts of observed outcomes


### Implementing a Uniform Distribution
- Goal is equal probability for all outcomes
  - apply H to every qubit so result is equal magnitude for all amplitudes
- In simulation, this works for any number of qubits using a loop


In [ ]:
q = QuantumRegister(3)
qc = QuantumCircuit(q)

for i in range(len(q)):
    qc.h(q[i])

state = qc.run()
print_state_table(state)
samples = measure(state, 1000)
print(samples)

### Day 3 Homework: Implement a Binomial Distribution
- Model number of successes in n trials
- Apply RY(θ) to each qubit
- Result:
  - amplitudes depend on number of 1s in outcome

- Uniform Distribution:
  - Every outcome has the same chance
  - Example (3 qubits):
  - 000, 001, …, 111 all have probability 1/8
- Binomial Distribution:
  - Probabilities depend on how many successes occur
  - Outcomes with the same number of successes share probability  
  - Example (3 trials, p = 0.5):
    - 0 successes: 1/8
    - 1 success: 3/8
    - 2 successes: 3/8
    - 3 successes: 1/8

In [ ]:
# Goal: create a non-uniform distribution across all 3 qubits using rotation gates
# 1. initialize a 3-qubit register and circuit
# 2. choose an angle θ for the RY rotation
# 3. loop over all qubits and apply RY(θ) to each one
# 4. run the circuit to produce the final state
# 5. print the state table to view amplitudes
# 6. measure the state using multiple shots
# 7. print the measurement counts
# 8. change θ and repeat to compare results


## References

- Gonciulea, C., & Stefanski, C. (2025). Building Quantum Software With Python: A Developer’s Guide. Manning Publications. ISBN: 978-1633437630.
- École Polytechnique Fédérale de Lausanne. (n.d.). IBM’s quantum computer links two quantum revolutions. https://actu.epfl.ch/news/ibm-s-quantum-computer-links-two-quantum-revolutio/
